# P1-A8: Semantic Relationships and Retrieval

**QuadConnect · The Connectors · Team 13 · INFO 490, Fall 2026**

A retrieval-augmented generation (RAG) system that answers students' questions about QuadConnect from its student
handbook. It turns the handbook into chunks, embeds them, retrieves the chunks closest to a question, and lets a
language model answer from them:

**handbook → chunks → embeddings → vector store → top-k chunks → prompt → language model → answer**

| Part | What it does | Status |
|---|---|---|
| 1. Build the RAG pipeline | The knowledge base, three chunking strategies, three embedding models, retrieval and generation | In progress (Steps 1.1 to 1.1) |
| 2. System experiments | 3 embedding models × 3 chunking strategies = 9 configurations, tested on realistic questions | Next |
| 3. Failure analysis and improvement | Failures, their causes, and one fix shown before and after | Next |
| 4. System design reflection | Cost, RAG against its alternatives, and a design for 10,000 users a day | Next |

This notebook is independent of the Django app: it only uses QuadConnect's handbook, in `data/`.

**To run:** `pip install -r requirements.txt` in this folder, then *Run All*. The first run downloads the three
embedding models from Hugging Face (about 1.9 GB). The language model's replies are saved in
`results/generations.jsonl`, so a re-run reuses them and needs no API key. A new question needs `GROQ_API_KEY` in
`../.env` (never committed). Every path is relative to this folder.

In [1]:
# Setup: imports, paths and the settings every step uses. Run this cell first.
import os
import platform
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*symlinks.*")  # Windows without Developer Mode copies files instead
warnings.filterwarnings("ignore", message="IProgress not found")  # progress bars need ipywidgets; not needed here

import pandas as pd
import sentence_transformers
import torch
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
DATA, RESULTS = Path("data"), Path("results")
RESULTS.mkdir(exist_ok=True)
load_dotenv(Path("..") / ".env")  # GROQ_API_KEY is needed only for a question whose answer is not saved yet
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or None

FIXED_WORDS = 120  # Step 1.2: the size of a fixed-length chunk, about one handbook paragraph
LONG_PARAGRAPH = 150  # Step 1.2: the hybrid strategy splits a paragraph longer than this
TOP_K = 3  # Step 1.4: chunks retrieved for each question
print(f"{platform.platform()} | {os.cpu_count()} logical CPUs | torch {torch.__version__} | "
      f"sentence-transformers {sentence_transformers.__version__}")
print(f"GROQ_API_KEY: {'set' if GROQ_API_KEY else 'not set (saved answers only)'}")


def require(condition, message):
    """Stop the notebook with the reason if a check fails."""
    if not condition:
        raise RuntimeError(message)

Windows-10-10.0.26300-SP0 | 16 logical CPUs | torch 2.14.1+cpu | sentence-transformers 6.2.0
GROQ_API_KEY: set


## Step 1.1: Define the knowledge base

**The dataset is QuadConnect's student handbook** (`data/handbook.md`): the help guide a student would read to
understand the app. It is the natural knowledge base for QuadConnect, because the questions students ask are about
how the app works: how matching decides, which times and places are possible, what happens to their feedback and
their data.

QuadConnect had no such document. Its database holds short fields (4 venues, 28 interests, 33 feedback notes), and
the repository's documents are written for developers. So we wrote the handbook for this assignment, from the app's
own code and pages. Every fact in it is one the app enforces or shows:

* the formats and preferences come from `connect/models.py`: Friend Connect (1-on-1) and Squad Connect (4 to 8
  students), the five-step social energy scale, weekends only in four two-hour windows, the 0–100 fit score and the
  check-in code;
* the venues, their seats and arrival notes, the interests and the colleges come from the seeded database;
* sign-in, cookies and data sharing come from the Privacy page and the settings, the venue check from
  `connect/ai_venues.py`, and search from the search pages.

**How the document is organised:** 13 sections, each about one subject, each made of paragraphs. A *paragraph* is a
block of text between blank lines under a `## ` section heading; headings are not paragraphs. Each paragraph covers
one idea, and the last section is a set of questions and answers, one question per paragraph. The table below checks
the brief's size requirements.

In [2]:
def read_handbook(path):
    """The handbook as paragraphs: one row per block of text between blank lines, under its section heading."""
    rows, section, number = [], None, 0
    for block in re.split(r"\n\s*\n", path.read_text(encoding="utf-8").strip()):
        block = " ".join(block.split())
        if block.startswith("## "):  # a section heading, such as "## 4. Setting your availability"
            section, number = block[3:], 0
            continue
        if block.startswith("# "):  # the document's title
            continue
        number += 1
        rows.append({"Paragraph": f"S{int(section.split('.')[0]):02d}-P{number}", "Section": section,
                     "Words": len(block.split()), "Text": block})
    return pd.DataFrame(rows)


handbook = read_handbook(DATA / "handbook.md")
words = handbook["Words"]
requirements = pd.DataFrame([
    ("Number of paragraphs", "at least 20; 30-50 recommended", f"{len(handbook)}", 30 <= len(handbook) <= 50),
    ("Paragraph length", "about 80-200 words each", f"{words.min()} to {words.max()} words (mean {words.mean():.0f})",
     words.between(80, 200).all()),
    ("Total size", "2,000-5,000 words", f"{words.sum():,} words", 2_000 <= words.sum() <= 5_000),
], columns=["Size requirement (brief, Step 1.2)", "The brief asks for", "The handbook has", "Met"])
display(requirements)
require(requirements["Met"].all(), "the knowledge base does not meet the brief's size requirements")

sections = handbook.groupby("Section", sort=False).agg(Paragraphs=("Paragraph", "count"), Words=("Words", "sum"))
display(sections)
print("One paragraph, as the pipeline sees it:\n")
print(handbook.loc[handbook["Paragraph"] == "S06-P2", ["Paragraph", "Section", "Words"]].to_string(index=False))
print(handbook.loc[handbook["Paragraph"] == "S06-P2", "Text"].iloc[0])

,"Size requirement (brief, Step 1.2)",The brief asks for,The handbook has,Met
0,Number of paragraphs,at least 20; 30-50 recommended,44,True
1,Paragraph length,about 80-200 words each,90 to 165 words (mean 111),True
2,Total size,"2,000-5,000 words","4,875 words",True


,Paragraphs,Words
Section,,
1. About QuadConnect,3,353
2. Creating an account and signing in,3,346
3. Building your profile,5,594
4. Setting your availability,2,222
5. How weekly matching works,4,473
"6. Meeting up: venues, check-in and icebreakers",4,504
7. Suggesting a new venue,4,466
8. Finding students,4,419
9. After the meetup: feedback,3,290


One paragraph, as the pipeline sees it:

Paragraph                                         Section  Words
   S06-P2 6. Meeting up: venues, check-in and icebreakers    165
Each venue page tells you how to find your group. It gives the street address, says whether the place is indoors or outdoors and how many people it seats, and adds an arrival note that says exactly where to wait. At the Illini Union you meet at the main entrance, in the ground floor lobby. At the Grainger Engineering Library you meet in the first floor commons. At the Main Quad, the only outdoor venue, you meet at the south end, by the Alma Mater statue. At Espresso Royale on Goodwin you meet in the counter seating area. The Illini Union seats twelve people, the Grainger Engineering Library eight, the Main Quad twenty and Espresso Royale six. A full squad of eight therefore fits every venue except Espresso Royale, which suits Friend Connect meetups and smaller squads. If you are not sure where to go, open the venue pa